# Get unique R1 and R2 pairs.

The sequencing files contains repeats of the same read. We aggregate them together and reserve the read counts.

In [ ]:
from naapam import align

align.remove_duplicates(
    data_dir="/home/ljw/sdc1/sx_data/SX/data", root_dir="/home/ljw/sdb1/naapam"
)

# Parse read components

The plasmid consists of U6 (the last 19 bps), G, sgRNA, scaffold, C, 44bp target sequence, CAG, barcode, 20bp primer. R1 sequence the plasmid from U6. R2 sequence the reverse-complement of the plasmid. Both R1 and R2 are prefixed by sequencing barcode. Because primer and scaffold are fixed across all plasmids, we first extract primer and scaffold from R1 and R2. By using the alignment library of biopython, we allow small mutations in primer and scaffold. For R1, the segment between primer and scaffold is sgRNA. For R2, that is barcode + CTG + target sequence.

In [ ]:
import subprocess

subprocess.run(args=["naapam-parse-parallel", "/home/ljw/sdb1/naapam"])

## Demultiplex barcode

Each plasmid has a unique barcode. We need to determine the best-matched barcode for each read. This is achieve through bowtie2.

### Index barcode

Build bowtie2 index for each barcode.

In [ ]:
from naapam import parse

parse.build_barcode(root_dir="/home/ljw/sdb1/naapam", plasmid_file=None)

### Prepare R2 segments containing barcode

Prepare R2 segments (barcode + CTG + target sequence) containing barcode ready to map by bowtie2 to barcodes.

In [ ]:
from naapam import parse

parse.prepare_barcode_CTG_target_prefix(root_dir="/home/ljw/sdb1/naapam")

### Map barcode

Map barcode + CTG + target sequence to barcodes.

In [ ]:
from naapam import parse

parse.map_barcode(root_dir="/home/ljw/sdb1/naapam")

### Parse barcode

Bowtie2 map each R2 segment to the best-matched barcode. We extract the match information including the map score from AS tag, the id of the bese-matched barcode, and the position of barcode in the R2 segment.

In [ ]:
from naapam import parse

parse.parse_barcode(root_dir="/home/ljw/sdb1/naapam")

## Demultiplex spacer sequence

Each plasmid has a unique spacer sequence. We need to determine the best-matched spacer sequence for each read. This is achieve through bowtie2.

### Index spacer sequence

Build bowtie2 index for each spacer sequence.

In [ ]:
from naapam import parse

parse.build_sgRNA(root_dir="/home/ljw/sdb1/naapam", plasmid_file=None)

### Prepare R1 spacer and R2 spacer

R1 contains spacer in sgRNA. R2 contains spacer in target. We prepare both to map to the sgRNA template in plasmids.

In [ ]:
from naapam import parse

parse.prepare_R1_sgRNA_and_R2_sgRNA(root_dir="/home/ljw/sdb1/naapam")

### Map spacer

Map R1 spacer and R2 spacer to the sgRNA template in plasmids.

In [ ]:
from naapam import parse

parse.map_sgRNA(root_dir="/home/ljw/sdb1/naapam")

### Parse spacer

Bowtie2 map each spacer to the best-matched reference spacer. We extract the match information including the map score from AS tag and the id of the bese-matched reference spacer.

In [ ]:
from naapam import parse

parse.parse_sgRNA(root_dir="/home/ljw/sdb1/naapam")

# Collect the parse results of controls

We collect the parse results of all controls and aggregate them by chips. We drop the R1 and R2 sequence barcodes to further merge the control sequence.

In [ ]:
from naapam import align

align.collect_control(root_dir="/home/ljw/sdb1/naapam")

# Calculate distribution for control reads

We calculate the distribution for the parse results, which are used for downstream filtering. In details, we hist the following values:
  - the R1 primer length;
  - the R1 sgRNA length;
  - the length of the scaffold sequenced in R1;
  - the length of the remained sequence downstream to scaffold in R1;
  - the R2 primer length;
  - the length of the sequence between R2 primer and barcode;
  - the length of the actual barcode in R2;
  - the length of the sequence between barcode and spacer in target sequence;
  - the length of actual spacer in target sequence;
  - the pam sequence;
  - the length of the sequence between pam and the reverse-complement scaffold sequenced in R2;
  - the length of the actual reverse-complement scaffold sequenced in R2;
  - the length of the remained sequence downstream to reverse-complement scaffold in R2;
  - the U6 start flag in R1 (should be G);
  - the U6 end flag in R2 (should be C because R2 sequence the reverse-complement strand);
  - the last two nucleotides in pam (GG or AA);
  - the alignment score between the actual R1 primer and the designed R1 one;
  - the alignment score between the actual scaffold sequenced in R1 and the designed one;
  - the alignment score between the actual R2 primer and the designed R2 one;
  - the alignment score between the spacer in R1 (sgRNA spacer) and that in R2 (target spacer);
  - the alignment score between the actual scaffold sequenced in R2 and the designed one;
  - the alignment score between the actual barcode and the designed one;
  - the alignment score between the actual spacer in R1 (sgRNA spacer) and the designed one;
  - the alignment score between the actual spacer in R2 (target spacer) and the designed one;
  - the id of the best-matched barcode;
  - the id of the best-matched spacer for R1 spacer (sgRNA spacer);
  - the id of the best-matched spacer for R2 spacer (target spacer);
  - the read count across all control samples.

In [ ]:
from naapam import align

align.stat_control(root_dir="/home/ljw/sdb1/naapam")

# Filter nonfunctional control

We filter control reads with abnormally short or long component length. We futher filter reads with low alignment score to the designed sequence (e.g. low quality primer, barcode, spacer and scaffold). We also require the correct U6 flags (G in R1 and C in R2). We then filter reads with low count across all control samples. We finally filter reads without match barcode/spacer or with inconsistant ids of barcode and spacer.

In [ ]:
from importlib import resources

import yaml

from naapam import align

with resources.as_file(
    resources.files("naapam.filter_configs") / "align.filter_nofunc_control.yaml"
) as pf:
    with pf.open("r") as fd:
        params = yaml.load(fd, Loader=yaml.CLoader)

align.filter_nofunc_control(root_dir="/home/ljw/sdb1/naapam", **params)

# Cluster control by mutant

We align control read to designed templated control sequence to get the mutant type of each read. To prevent the control reference from too diverse, we cluster control read by mutant types. In details, we neglect irrelevant small mutations away from cleavage site and only classify control read by the main editing patten at the cleavage site, including resection and templated insertion at both ends and the random insertion.

In [ ]:
from importlib import resources

import yaml

from naapam import align

with resources.as_file(
    resources.files("naapam.filter_configs")
    / "align.cluster_func_control_by_mutant.yaml"
) as pf:
    with pf.open("r") as fd:
        params = yaml.load(fd, Loader=yaml.CLoader)

align.cluster_func_control_by_mutant(
    root_dir="/home/ljw/sdb1/naapam", **params, plasmid_file=None
)

# Calculate distributions for control mutant types

Calculate distributions of control mutant types. This includes:
  - the number of mutant types per barcode id;
  - the upstream deletion size of the cleavage site;
  - the downstream deletion size of the cleavage site;
  - the random insertion size of the cleavage site;
  - the count of each mutant type;
  - the total count of each barcode id over all mutant types;
  - the wild-type ratio of each barcode id;
  - the ratio between the second most and the most among:
    - wild type;
    - 2bp upstream deletion and 1bp downstream templated insertion;
    - 3bp upstream deletion and 2bp downstream templated insertion;
    - 4bp upstream deletion and 3bp downstream templated insertion
    - 5bp upstream deletion and 4bp downstream templated insertion
  - ...

In [ ]:
from importlib import resources

import yaml

from naapam import align

with resources.as_file(
    resources.files("naapam.filter_configs")
    / "align.cluster_func_control_by_mutant.yaml"
) as pf:
    with pf.open("r") as fd:
        params = yaml.load(fd, Loader=yaml.CLoader)

align.stat_func_control(root_dir="/home/ljw/sdb1/naapam", **params)

# 筛掉低质量barcode

In [ ]:
from importlib import resources

import yaml

from naapam import align

with resources.as_file(
    resources.files("naapam.filter_configs") / "align.filter_low_quality_barcode.yaml"
) as pf:
    with pf.open("r") as fd:
        params = yaml.load(fd, Loader=yaml.CLoader)

align.filter_low_quality_barcode(root_dir="/home/ljw/sdb1/naapam", **params)

# 筛掉低质量mutant

In [ ]:
from importlib import resources

import yaml

from naapam import align

with resources.as_file(
    resources.files("naapam.filter_configs") / "align.filter_low_quality_mutant.yaml"
) as pf:
    with pf.open("r") as fd:
        params = yaml.load(fd, Loader=yaml.CLoader)

align.filter_low_quality_mutant(root_dir="/home/ljw/sdb1/naapam", **params)

# 生成参考序列

In [ ]:
from importlib import resources

import yaml

from naapam import align

with resources.as_file(
    resources.files("naapam.filter_configs") / "align.generate_reference.yaml"
) as pf:
    with pf.open("r") as fd:
        params = yaml.load(fd, Loader=yaml.CLoader)

align.generate_reference(root_dir="/home/ljw/sdb1/naapam", **params)

# 收集treat的parse结果

In [ ]:
from naapam import align

align.collect_treat(root_dir="/home/ljw/sdb1/naapam")

# 统计treat指标

In [ ]:
from naapam import align

align.stat_treat(root_dir="/home/ljw/sdb1/naapam")

# 筛选treat

In [ ]:
from importlib import resources

import yaml

from naapam import align

with resources.as_file(
    resources.files("naapam.filter_configs") / "align.filter_treat.yaml"
) as pf:
    with pf.open("r") as fd:
        params = yaml.load(fd, Loader=yaml.CLoader)

align.filter_treat(root_dir="/home/ljw/sdb1/naapam", **params)

# 解耦合

In [ ]:
from naapam import align

align.demultiplex(root_dir="/home/ljw/sdb1/naapam")

# 匹配

In [ ]:
import subprocess

subprocess.run(args=["naapam-align-parallel", "/home/ljw/sdb1/naapam"])